# Predicciones sobre test — TP Grupo 10

## Imports y rutas

In [1]:
import os
import numpy as np
import pandas as pd
from joblib import load
from sklearn.metrics import cohen_kappa_score, accuracy_score, balanced_accuracy_score, confusion_matrix
from sklearn.preprocessing import MinMaxScaler

In [2]:
BASE_DIR = os.path.join(os.getcwd(), '..', '..')

PATH_TRAIN_CLEAN = os.path.join(BASE_DIR, 'work/cleaned/train_clean.csv')
PATH_TEST_CLEAN  = os.path.join(BASE_DIR, 'work/cleaned/test_clean.csv')
PATH_MODEL       = os.path.join(BASE_DIR, 'work/models/lgbm_final_tp10.joblib')

FEATURES_SEL = [
    'age_x_MaturitySize', 'Age', 'PhotoAmt', 'health_score',
    'health_complete', 'rescuer_listings', 'Breed1', 'Sterilized',
    'Breed2', 'photo_video_score'
]

## Carga de datos

In [3]:
train = pd.read_csv(PATH_TRAIN_CLEAN)
test  = pd.read_csv(PATH_TEST_CLEAN)
print(f'Train: {train.shape} | Test: {test.shape}')

Train: (11994, 26) | Test: (2999, 26)


## Feature Engineering

In [4]:
def build_features(df, scaler=None, rescuer_counts=None, fit=True):
    X = df.drop(columns=['AdoptionSpeed', 'PetID', 'RescuerID', 'Name', 'Description'])
    X['health_score'] = (
        (df['Vaccinated'] == 1).astype(int) +
        (df['Dewormed']   == 1).astype(int) +
        (df['Sterilized'] == 1).astype(int) +
        (df['Health']     == 1).astype(int)
    )
    X['health_complete'] = (X['health_score'] == 4).astype(int)
    X['age_x_MaturitySize'] = (X['Age'] / 12) * X['MaturitySize']
    if fit:
        scaler = MinMaxScaler()
        X['age_x_MaturitySize'] = scaler.fit_transform(X[['age_x_MaturitySize']])
    else:
        X['age_x_MaturitySize'] = scaler.transform(X[['age_x_MaturitySize']])
    X['photo_video_score'] = df['PhotoAmt'] + df['VideoAmt'] * 2
    if fit:
        rescuer_counts = df['RescuerID'].value_counts()
        X['rescuer_listings'] = df['RescuerID'].map(rescuer_counts)
    else:
        X['rescuer_listings'] = df['RescuerID'].map(rescuer_counts).fillna(1)
    return X[FEATURES_SEL], scaler, rescuer_counts

_, scaler, rescuer_counts = build_features(train, fit=True)
X_test, _, _ = build_features(test, scaler=scaler, rescuer_counts=rescuer_counts, fit=False)
y_test = test['AdoptionSpeed']
print(f'X_test: {X_test.shape}')

X_test: (2999, 10)


## Cargar modelo y predecir

In [5]:
model = load(PATH_MODEL)
predicciones = model.predict(X_test)
print('Predicciones generadas.')

C:\anaconda\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.8.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Predicciones generadas.


## Métricas

In [6]:
kappa   = cohen_kappa_score(y_test, predicciones, weights='quadratic')
acc     = accuracy_score(y_test, predicciones)
bal_acc = balanced_accuracy_score(y_test, predicciones)

print('=== Métricas sobre test ===')
print(f'  Cohen Kappa (quadratic): {kappa:.4f}')
print(f'  Accuracy:                {acc:.4f}')
print(f'  Balanced Accuracy:       {bal_acc:.4f}')

=== Métricas sobre test ===
  Cohen Kappa (quadratic): 0.2928
  Accuracy:                0.3938
  Balanced Accuracy:       0.3634


## Resultados

In [7]:
resultado = test[['PetID']].copy()
resultado['AdoptionSpeed_real']     = y_test.values
resultado['AdoptionSpeed_predicho'] = predicciones

out_path = os.path.join(os.getcwd(), 'predicciones_test.csv')
resultado.to_csv(out_path, index=False)
print(f'Predicciones guardadas en: {out_path}')
resultado.head(10)

Predicciones guardadas en: C:\Users\Hp EliteBook\Documents\GitHub\UA_MDM_Labo2_Grupo10\delfina\mica\predicciones_test.csv


,PetID,AdoptionSpeed_real,AdoptionSpeed_predicho
0,8f20e24ef,4,2
1,2d72ef0c4,4,1
2,44cd12263,4,2
3,210c4a637,2,2
4,21493e6ea,4,3
5,65ba8b6a0,4,0
6,bef935312,1,1
7,157e3e074,2,2
8,0ac71c2c1,1,4
9,128293a4d,4,4
